In [1]:
print("kernel ready")

kernel ready


# Section 1: Ground work

##### - Setting up the API key

##### 1. https://jooble.org/api/about - get your api key from here 
##### 2. Create a .env file and save your api key in the file

##### - Create virtual environment 

| Tool     | macOS                                                  | Windows                                            |
| -------- | ------------------------------------------------------ | -------------------------------------------------- |
| **venv** | `python3 -m venv .venv`<br>`source .venv/bin/activate` | `python -m venv .venv`<br>`.venv\Scripts\activate` |
| **uv**   | `uv venv`<br>`source .venv/bin/activate`               | `uv venv`<br>`.venv\Scripts\activate`              |

##### - install the requirements from requirements.txt
pip install -r requirements.txt



# Section 2: Fetching the data from the API

In [2]:
import time
import requests
import pandas as pd
from bs4 import BeautifulSoup
from typing import Any
from dotenv import load_dotenv
import os
import importlib

load_dotenv()


True

In [3]:
API_KEY = os.getenv("api_key") # get the api key from the env

url = f"https://jooble.org/api/{API_KEY}"
data = " " # intialising an empty variable to store the data

payload = {
    "keywords": "AI ML Engineer",
    "location": "India",
    "page": "1"
}

response = requests.post(url, json=payload)

if response.status_code != 200:
    print(f"[!] Request failed with status code: {response.status_code}")
else:
  jobs_list = response.json() # response back from the url


if jobs_list:
  print(jobs_list)

  # # Look at total count and a sample job returned
  print("Total jobs found by Jooble:", jobs_list.get("totalCount"))

  single_job = jobs_list.get("jobs")[0]
  print("\nA single jobs looks like \n", )
  display(single_job)

{'totalCount': 313, 'jobs': [{'title': 'AI Engineer', 'location': 'India', 'snippet': '&nbsp;...governments realize their greatest potential.  Title and Summary  <b>AI Engineer </b>\r\n Job Description Summary \r\n We believe in power of data. We...&nbsp;&nbsp;..., and cloud AI platforms. \r\n• Develop and maintain scalable AI/<b>ML </b>pipelines, data preparation workflows, and cloud-native...&nbsp;', 'salary': '', 'source': 'decentrajobs.com', 'type': 'Full-time', 'link': 'https://jooble.org/jdp/-4752801929770794022', 'company': 'Mastercard', 'updated': '2026-08-05T08:16:31.0070000', 'id': -4752801929770794022}, {'title': 'AI Engineer', 'location': 'India', 'snippet': '&nbsp;...governments realize their greatest potential.  Title and Summary  <b>AI Engineer </b>\r\n Job Description \r\n Our Purpose \r\n Mastercard powers...&nbsp;&nbsp;...compliance. \r\n Key Responsibilities \r\n• Design, develop, and deploy AI/<b>ML,</b> Generative AI, and Agentic AI solutions to solve complex...&nb

{'title': 'AI Engineer',
 'location': 'India',
 'snippet': '&nbsp;...governments realize their greatest potential.  Title and Summary  <b>AI Engineer </b>\r\n Job Description Summary \r\n We believe in power of data. We...&nbsp;&nbsp;..., and cloud AI platforms. \r\n• Develop and maintain scalable AI/<b>ML </b>pipelines, data preparation workflows, and cloud-native...&nbsp;',
 'salary': '',
 'source': 'decentrajobs.com',
 'type': 'Full-time',
 'link': 'https://jooble.org/jdp/-4752801929770794022',
 'company': 'Mastercard',
 'updated': '2026-08-05T08:16:31.0070000',
 'id': -4752801929770794022}

# Section 3: Cleaning the fetched data for downstream activities

In [4]:
import pandas as pd

def clean_html(raw_html: str) -> str:
    """Strips HTML tags and normalizes whitespace."""
    if not raw_html:
        return ""
    soup = BeautifulSoup(raw_html, "html.parser")
    # Replace line breaks and tags with clean space
    text = soup.get_text(separator=" ")
    return " ".join(text.split())

jobs_df = pd.DataFrame(jobs_list.get("jobs"))

# Select only the columns relevant for our pipeline
jobs_df = jobs_df[["id", "title", "company", "location", "salary", "link", "snippet"]]
jobs_df.head(3)

,id,title,company,location,salary,link,snippet
0,-4752801929770794022,AI Engineer,Mastercard,India,,https://jooble.org/jdp/-4752801929770794022,&nbsp;...governments realize their greatest po...
1,-7451179239851005482,AI Engineer,Mastercard,India,,https://jooble.org/jdp/-7451179239851005482,&nbsp;...governments realize their greatest po...
2,-5443742953511610657,AI Engineer - Computer Vision,EveryWatch,India,,https://jooble.org/jdp/-5443742953511610657,&nbsp;...commitment and creativity in building...


In [5]:
# Apply the cleaning function to the snippet column

jobs_df["clean_description"] = jobs_df["snippet"].apply(clean_html)

# Show before vs after for comparison
print("BEFORE (Raw HTML):")
print(jobs_df["snippet"].iloc[0][:150])
print("\nAFTER (Clean Text):")
print(jobs_df["clean_description"].iloc[0][:150])

BEFORE (Raw HTML):
&nbsp;...governments realize their greatest potential.  Title and Summary  <b>AI Engineer </b>
 Job Description Summary 
 We believe in power of dat

AFTER (Clean Text):
...governments realize their greatest potential. Title and Summary AI Engineer Job Description Summary We believe in power of data. We... ..., and clo


# Section 4: Fetching data from multiple pages

In [6]:
# lets loop over multiple pages

def fetch_jobs_pipeline(api_key, keywords="AI ML Engineer", location="India", total_pages=3):
    url = f"https://jooble.org/api/{api_key}"
    all_jobs = []

    for page in range(1, total_pages + 1):
        payload = {
            "keywords": keywords,
            "location": location,
            "page": str(page)
        }

        response = requests.post(url, json=payload)

        if response.status_code == 200:
            jobs = response.json().get("jobs", [])
            all_jobs.extend(jobs)
            print(f"✅ Fetched page {page} ({len(jobs)} jobs)")
        else:
            print(f"❌ Failed to fetch page {page}")
            break

        time.sleep(0.5) # Politeness delay

    # Build and clean DataFrame
    df = pd.DataFrame(all_jobs)
    if not df.empty:
        df = df[["id", "title", "company", "location", "salary", "link", "snippet"]]
        df["clean_description"] = df["snippet"].apply(clean_html)
        df = df.drop_duplicates(subset=["id"]).reset_index(drop=True)

    return all_jobs , df

In [7]:
all_jobs , jobs_df = fetch_jobs_pipeline(API_KEY, keywords="AI ML Engineer", location="India", total_pages=3)

print(jobs_df.shape)
jobs_df.head(3)

✅ Fetched page 1 (30 jobs)
✅ Fetched page 2 (30 jobs)
✅ Fetched page 3 (30 jobs)
(90, 8)


,id,title,company,location,salary,link,snippet,clean_description
0,-4752801929770794022,AI Engineer,Mastercard,India,,https://jooble.org/jdp/-4752801929770794022,&nbsp;...governments realize their greatest po...,...governments realize their greatest potentia...
1,-7451179239851005482,AI Engineer,Mastercard,India,,https://jooble.org/jdp/-7451179239851005482,&nbsp;...governments realize their greatest po...,...governments realize their greatest potentia...
2,-5443742953511610657,AI Engineer - Computer Vision,EveryWatch,India,,https://jooble.org/jdp/-5443742953511610657,&nbsp;...commitment and creativity in building...,...commitment and creativity in building real-...


In [8]:
# all_ids = [job["id"] for job in all_jobs]
# all_ids[:5]

In [9]:
jobs_df_req = jobs_df.drop(columns = ["snippet"])
jobs_df_req.head(3)

,id,title,company,location,salary,link,clean_description
0,-4752801929770794022,AI Engineer,Mastercard,India,,https://jooble.org/jdp/-4752801929770794022,...governments realize their greatest potentia...
1,-7451179239851005482,AI Engineer,Mastercard,India,,https://jooble.org/jdp/-7451179239851005482,...governments realize their greatest potentia...
2,-5443742953511610657,AI Engineer - Computer Vision,EveryWatch,India,,https://jooble.org/jdp/-5443742953511610657,...commitment and creativity in building real-...


# Section 5: Knowledge Check

![kc](./asset/kc.png)

# Section 6: Parsing Resume - pdf and docx files

In [12]:
# !pip install pdfplumber python-docx

In [13]:
import io
import pdfplumber
import docx

def parse_resume(file_input):
    """
    Extracts text from PDF, DOCX, or TXT files.
    Accepts a filepath string or an in-memory uploaded file buffer.
    """
    # 1. Determine file extension
    file_name = file_input.name if hasattr(file_input, "name") else str(file_input)
    ext = file_name.split(".")[-1].lower()

    extracted_text = []

    # 2. Extract PDF using pdfplumber
    if ext == "pdf":
        # Handle both file paths and byte streams (Streamlit upload)
        with pdfplumber.open(file_input) as pdf:
            for page in pdf.pages:
                text = page.extract_text()
                if text:
                    extracted_text.append(text)

    # 3. Extract Word DOCX using python-docx
    elif ext == "docx":
        # python-docx requires a BytesIO stream if reading in-memory bytes
        if hasattr(file_input, "read"):
            doc = docx.Document(io.BytesIO(file_input.read()))
        else:
            doc = docx.Document(file_input)

        for paragraph in doc.paragraphs:
            if paragraph.text.strip():
                extracted_text.append(paragraph.text)

    # 4. Fallback for plain text files
    elif ext == "txt":
        if hasattr(file_input, "read"):
            return file_input.read().decode("utf-8")
        with open(file_input, "r", encoding="utf-8") as f:
            return f.read()

    else:
        raise ValueError(f"Unsupported file format: .{ext}. Please provide a .pdf or .docx file.")

    full_text = "\n".join(extracted_text)
    return " ".join(full_text.split())  # Clean extra whitespace

In [14]:
# Test with a local file path
sample_resume_text = parse_resume("./asset/Eshika-Mahajan-Resume.pdf")
# sample_resume_text = parse_resume("./asset/Eshika-Mahajan-Resume.docx")

print(f"✅ Extracted {len(sample_resume_text)} characters.")
print("\nFirst 300 characters of parsed resume:")
print(sample_resume_text[:300])

✅ Extracted 7015 characters.

First 300 characters of parsed resume:
Eshika Mahajan Proficiency Certification AI ML • AWS Certified AI Practitioner • LLMs, Open AI API, Gemini, LangChain, LangGraph, RAG, Vector • AWS Certified Machine Learning Search, Tools – Function Calling, MCP Servers, Guardrails, Traditional Specialty ML, DL, NLP, ML Ops • AWS Certified Cloud Pr


# Section 7 : 

![TF-IDF explained](./asset/tf-idf.png)

In [15]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np

df_jobs = jobs_df.copy()  # Use the DataFrame created from the API fetch

# 1. Combine resume (document 0) + all job descriptions (documents 1 to N)
# 'sample_resume_text' is the string extracted in the previous pdfplumber step
documents = [sample_resume_text] + df_jobs["clean_description"].tolist()

# 2. Build the vocabulary and compute TF-IDF weights
# ngram_range=(1, 2) captures single words ("python") and pairs ("machine learning")
vectorizer = TfidfVectorizer(stop_words="english", ngram_range=(1, 2))
tfidf_matrix = vectorizer.fit_transform(documents)

print(f"Matrix shape: {tfidf_matrix.shape}")
print(f"Total documents: {tfidf_matrix.shape[0]} (1 resume + {len(df_jobs)} jobs)")
print(f"Total unique n-grams extracted: {tfidf_matrix.shape[1]}")

# 3. Calculate Cosine Similarity
# Vector at index 0 (Resume) compared against vectors at indices 1 to end (All Jobs)
resume_vector = tfidf_matrix[0:1]
job_vectors = tfidf_matrix[1:]

scores = cosine_similarity(resume_vector, job_vectors).flatten()

print(f"\nSimilarity scores computed for {len(scores)} jobs.")
print(f"Top raw score: {np.max(scores):.4f} | Lowest raw score: {np.min(scores):.4f}")

Matrix shape: (91, 2830)
Total documents: 91 (1 resume + 90 jobs)
Total unique n-grams extracted: 2830

Similarity scores computed for 90 jobs.
Top raw score: 0.1192 | Lowest raw score: 0.0018


In [ ]:
def rank_jobs_tfidf(resume_text: str, df: pd.DataFrame, top_n: int = 10) -> pd.DataFrame:
    """
    Ranks job listings against a resume string using TF-IDF and Cosine Similarity.
    Returns the top_n most relevant jobs with similarity percentages.
    """
    # Create combined corpus
    corpus = [resume_text] + df["clean_description"].tolist()
    
    # Vectorize
    vectorizer = TfidfVectorizer(stop_words="english", ngram_range=(1, 2))
    tfidf_matrix = vectorizer.fit_transform(corpus)
    
    # Compute similarity between resume (row 0) and all jobs (rows 1:)
    similarity_scores = cosine_similarity(tfidf_matrix[0:1], tfidf_matrix[1:]).flatten()
    
    # Create a copy and append scores
    df_ranked = df.copy()
    df_ranked["tfidf_score"] = similarity_scores
    df_ranked["match_score_%"] = (df_ranked["tfidf_score"] * 100).round(1)
    
    # Sort descending and return top_n
    return df_ranked.sort_values(by="tfidf_score", ascending=False).head(top_n).reset_index(drop=True)

# ---------------------------------------------------------
# Run and inspect the top 10 matches
# ---------------------------------------------------------
top_10_jobs = rank_jobs_tfidf(sample_resume_text, df_jobs, top_n=10)

# Display the ranked results to show role separation
top_10_jobs[["title", "company", "match_score_%"]]

![Why Raw Similarity Score Is Low](./asset/low_tf-idf_ranks.png)

In [17]:
# Create a richer text representation
df_jobs["enriched_text"] = df_jobs["title"] + " " + df_jobs["clean_description"]

corpus = [sample_resume_text] + df_jobs["enriched_text"].tolist()


from sklearn.preprocessing import MinMaxScaler
import numpy as np

# Scale raw scores from 0% to 100% relative to the best match in the batch
scaler = MinMaxScaler(feature_range=(20, 95)) # Keep baseline at 20%, top at 95%
normalized_scores = scaler.fit_transform(scores.reshape(-1, 1)).flatten()

df_jobs["relative_match_%"] = np.round(normalized_scores, 1)

df_jobs

,id,title,company,location,salary,link,snippet,clean_description,enriched_text,relative_match_%
0,-4752801929770794022,AI Engineer,Mastercard,India,,https://jooble.org/jdp/-4752801929770794022,&nbsp;...governments realize their greatest po...,...governments realize their greatest potentia...,AI Engineer ...governments realize their great...,90.8
1,-7451179239851005482,AI Engineer,Mastercard,India,,https://jooble.org/jdp/-7451179239851005482,&nbsp;...governments realize their greatest po...,...governments realize their greatest potentia...,AI Engineer ...governments realize their great...,68.5
2,-5443742953511610657,AI Engineer - Computer Vision,EveryWatch,India,,https://jooble.org/jdp/-5443742953511610657,&nbsp;...commitment and creativity in building...,...commitment and creativity in building real-...,AI Engineer - Computer Vision ...commitment an...,36.5
3,7852742555688528419,Senior AI Engineer,Pear Tree,India,,https://jooble.org/jdp/7852742555688528419,We are seeking an experienced and innovative S...,We are seeking an experienced and innovative S...,Senior AI Engineer We are seeking an experienc...,34.3
4,-8399419085026992319,Code Repository Licensing for AI Training - So...,Gramian Consulting,India,,https://jooble.org/jdp/-8399419085026992319,&nbsp;...boutique consultancy specializing in ...,...boutique consultancy specializing in IT pro...,Code Repository Licensing for AI Training - So...,40.5
...,...,...,...,...,...,...,...,...,...,...
85,-3998318708916043517,Lead Engineer - Mediation,Dish,India,,https://jooble.org/jdp/-3998318708916043517,&nbsp;...facilities in India are some of EchoS...,...facilities in India are some of EchoStar's ...,Lead Engineer - Mediation ...facilities in Ind...,31.2
86,2856281270334230642,Lead Software Engineer,Mastercard,India,,https://jooble.org/jdp/2856281270334230642,&nbsp;...realize their greatest potential. Ti...,...realize their greatest potential. Title and...,Lead Software Engineer ...realize their greate...,45.0
87,-6205394815572823551,Staff Software Engineer,CertifyOS,India,,https://jooble.org/jdp/-6205394815572823551,About CertifyOS CertifyOS is building the data...,About CertifyOS CertifyOS is building the data...,Staff Software Engineer About CertifyOS Certif...,42.4
88,-2151720382784937269,Lead BizOps Engineer,Mastercard,India,,https://jooble.org/jdp/-2151720382784937269,&nbsp;...products and services that help peopl...,"...products and services that help people, bus...",Lead BizOps Engineer ...products and services ...,23.4


#### 💡 Notice how our statistical model (TF-IDF) gives a top score of only 12% because it relies on exact string overlap. 
<p> This is where classical NLP hits a wall.<br>Next, we hand these top 10 candidates over to an LLM (Gemini) to evaluate actual conceptual and semantic fit.</p>

# Section: GET LLM RESPONSE

In [18]:
import json

load_dotenv(override=True)

from google import genai
from google.genai import types
from structured_output_helper import MatchReport , JobEvaluation

# Initialize Gemini Client (reads GEMINI_API_KEY from env, or pass directly: api_key="...")
GEMINI_API_KEY = os.getenv("GEMINI_API_KEY")  # Ensure this is set in your environment
client = genai.Client(api_key=GEMINI_API_KEY)

GEMINI_API_KEY[:3]

'AQ.'

In [19]:
def evaluate_matches_with_gemini(
    resume_text: str, 
    top_10_df: pd.DataFrame
) -> MatchReport:
    """
    Passes top TF-IDF candidate jobs and resume text into Gemini for deep semantic evaluation.
    Enforces structured output via Pydantic.
    """
    # Prepare a compact payload of the top 10 jobs to minimize token consumption
    jobs_payload = top_10_df[["id", "title", "company", "clean_description"]].to_dict(orient="records")
    
    prompt = f"""
    You are an expert AI Technical Recruiter. Evaluate the candidate's resume against the 
    provided shortlist of candidate jobs (pre-filtered by relevance).

    RESUME TEXT:
    ---
    {resume_text[:4000]}
    ---

    CANDIDATE JOBS LIST (JSON):
    ---
    {json.dumps(jobs_payload, indent=2)}
    ---

    Task:
    1. Assess semantic fit (look beyond exact keywords for conceptual overlap like PyTorch vs Deep Learning).
    2. Score fit from 0 to 100.
    3. Identify matching skills and missing technical requirements.
    4. Provide a punchy 2-sentence rationale per role.
    """

    # Call Gemini with structured output constraint
    response = client.models.generate_content(
        model="gemini-3.6-flash",
        contents=prompt,
        config=types.GenerateContentConfig(
            response_mime_type="application/json",
            response_schema=MatchReport,
            temperature=0.2, # Low temperature for deterministic, analytical consistency
        ),
    )

    # response.parsed is automatically an instance of MatchReport
    return response.parsed

In [20]:
# Run evaluation on the top 10 DataFrame obtained from Cell 10
report: MatchReport = evaluate_matches_with_gemini(sample_resume_text, top_10_jobs)

print(f"Candidate Profile Summary: {report.summary}\n")
print(f"Total Evaluated Jobs: {len(report.evaluated_jobs)}\n" + "="*60)

# Inspect individual evaluations
for match in report.evaluated_jobs[:3]:
    print(f"\n🏢 Role: {match.title} at {match.company}")
    print(f"📊 Semantic Score: {match.semantic_fit_score}%")
    print(f"✅ Matching Skills: {', '.join(match.matching_skills)}")
    print(f"⚠️ Skill Gaps: {', '.join(match.missing_skills)}")
    print(f"💡 Rationale: {match.rationale}")
    print("-" * 60)

Direct use of automatic function calling (AFC) in Models.generate_content is not recommended. Instead, we recommend to use AFC in Chat.send_message. Similarly, direct use of AFC in Models.generate_content_stream is not recommended. Instead, we recommend to use AFC in Chat.send_message_stream.


Candidate Profile Summary: Eshika is an accomplished AI/ML Engineer with deep expertise in Agentic AI, Generative AI, LLM orchestration (LangChain/LangGraph, MCP), and production ML deployment across AWS, Azure, and GCP.

Total Evaluated Jobs: 10

🏢 Role: Lead Software Engineer at Mastercard
📊 Semantic Score: 78%
✅ Matching Skills: AWS, Data Engineering, PySpark, Python
⚠️ Skill Gaps: Databricks, Lead Architecture Experience
💡 Rationale: Eshika has strong experience with AWS ML/cloud services and data pipeline engineering using PySpark. However, the role emphasizes Databricks and lead-level data platform architecture which she partially lacks.
------------------------------------------------------------

🏢 Role: AI Engineer at Mastercard
📊 Semantic Score: 92%
✅ Matching Skills: AI/ML Pipelines, Cloud AI Platforms, Python, GCP, Azure, AWS
⚠️ Skill Gaps: Databricks
💡 Rationale: Eshika's background in building production-scale AI pipelines across AWS, Azure, and GCP fits this role excepti

In [22]:
# Convert Pydantic objects into a clean comparison DataFrame
evaluated_records = [job.model_dump() for job in report.evaluated_jobs]
df_results = pd.DataFrame(evaluated_records)

# Sort by Gemini's semantic score
df_results = df_results.sort_values(by="semantic_fit_score", ascending=False).reset_index(drop=True)

df_results[["title", "company", "semantic_fit_score", "matching_skills", "missing_skills"]].head()

,title,company,semantic_fit_score,matching_skills,missing_skills
0,AI Engineer,Mastercard,95,"[Agentic AI, Generative AI, LLMs, AI/ML Deploy...",[]
1,LLM Systems / AI Agent Engineer,Smartworking,94,"[AI Agents, Agentic AI, LLMs, LangChain, LangG...",[]
2,AI Engineer,Mastercard,92,"[AI/ML Pipelines, Cloud AI Platforms, Python, ...",[Databricks]
3,"Senior AI Engineer, AI Engineering",Mastercard,85,"[ML Engineering, AI Strategy, Python, Cloud Pl...",[Team Leadership Experience]
4,Principal Applied AI App Engineer,Mastercard,82,"[LLMs, Generative AI, Model Integration, APIs,...","[Principal-level Leadership, Enterprise System..."


![Observe](./asset/observe.png)

In [28]:
# from skill_recommender import UpskillingRoadmap , get_course_url
from skill_recommender import UpskillingRoadmap , get_course_url

import time
from google.genai.errors import ServerError

def generate_upskilling_roadmap(
    row_index: int, 
    df: pd.DataFrame, 
    client: genai.Client,
    max_retries: int = 3
) -> UpskillingRoadmap:
    """
    Picks a job row from df_results, inspects its missing_skills,
    and uses Gemini to recommend a 3-tier course roadmap per skill.
    Includes automated retries for transient 503 server hiccups.
    """
    selected_job = df.iloc[row_index]
    missing_skills = selected_job["missing_skills"]
    
    if not missing_skills:
        print(f"🎉 No missing skills for '{selected_job['title']}' at {selected_job['company']}!")
        return None

    prompt = f"""
    You are an AI Career Coach. A candidate is applying for the role:
    - Role: {selected_job['title']}
    - Company: {selected_job['company']}
    
    The candidate has identified the following missing skill gaps:
    {missing_skills}

    For EACH missing skill listed above, provide exactly 3 well-known, high-quality, 
    real-world learning resources:
    1. Beginner (foundations & core mental models)
    2. Intermediate (hands-on application & tooling)
    3. Advanced (enterprise scaling, production patterns, or leadership)
    """

    for attempt in range(1, max_retries + 1):
        try:
            response = client.models.generate_content(
                model="gemini-3.6-flash",
                contents=prompt,
                config=types.GenerateContentConfig(
                    response_mime_type="application/json",
                    response_schema=UpskillingRoadmap,
                    temperature=0.2,
                ),
            )
            return response.parsed

        except ServerError as err:
            if attempt < max_retries:
                wait_time = attempt * 2
                print(f"⚠️ Service busy (503). Retrying attempt {attempt}/{max_retries} in {wait_time}s...")
                time.sleep(wait_time)
            else:
                print(f"❌ Failed after {max_retries} attempts.")
                raise err

In [29]:
ROW_IDX = 3

roadmap: UpskillingRoadmap = generate_upskilling_roadmap(
    row_index=ROW_IDX, 
    df=df_results, 
    client=client
)

if roadmap:
    print(f"🎯 Target: {roadmap.target_role} at {roadmap.company}\n" + "=" * 65)
    for path in roadmap.learning_paths:
        print(f"\n📚 Skill: {path.skill_name.upper()}")
        for course in path.courses:
            url = get_course_url(course)
            print(f"  [{course.level}] {course.course_name} ({course.platform_or_provider})")
            print(f"   ↳ Link: {url}")
            print(f"   ↳ Focus: {course.key_takeaway}\n")
        print("-" * 65)

🎯 Target: Senior AI Engineer, AI Engineering at Mastercard

📚 Skill: TEAM LEADERSHIP EXPERIENCE
  [Beginner] Leadership Foundations (LinkedIn Learning)
   ↳ Link: https://www.google.com/search?q=Leadership+Foundations+LinkedIn+Learning
   ↳ Focus: Master core leadership styles, active listening, and foundational communication techniques essential for leading technical teams.

  [Intermediate] Engineering Leadership & Team Management (Coursera)
   ↳ Link: https://www.coursera.org/search?query=Engineering+Leadership+%26+Team+Management
   ↳ Focus: Learn how to delegate tasks, run effective technical sprints, and foster collaboration across cross-functional engineering units.

  [Advanced] The Elegant Puzzle: Systems of Engineering Management (O'Reilly Media)
   ↳ Link: https://www.google.com/search?q=The+Elegant+Puzzle%3A+Systems+of+Engineering+Management+O%27Reilly+Media
   ↳ Focus: Gain mastery over engineering organizational design, strategic resource allocation, and scaling high-perf

In [30]:
ROW_IDX = 4

roadmap: UpskillingRoadmap = generate_upskilling_roadmap(
    row_index=ROW_IDX, 
    df=df_results, 
    client=client
)

if roadmap:
    print(f"🎯 Target: {roadmap.target_role} at {roadmap.company}\n" + "=" * 65)
    for path in roadmap.learning_paths:
        print(f"\n📚 Skill: {path.skill_name.upper()}")
        for course in path.courses:
            url = get_course_url(course)
            print(f"  [{course.level}] {course.course_name} ({course.platform_or_provider})")
            print(f"   ↳ Link: {url}")
            print(f"   ↳ Focus: {course.key_takeaway}\n")
        print("-" * 65)

🎯 Target: Principal Applied AI App Engineer at Mastercard

📚 Skill: PRINCIPAL-LEVEL LEADERSHIP
  [Beginner] Engineering Leadership Fundamentals (Coursera)
   ↳ Link: https://www.coursera.org/search?query=Engineering+Leadership+Fundamentals
   ↳ Focus: Understand core mental models for leading technical teams and driving engineering alignment.

  [Intermediate] The Staff Engineer's Path (O'Reilly)
   ↳ Link: https://www.google.com/search?q=The+Staff+Engineer%27s+Path+O%27Reilly
   ↳ Focus: Master technical leadership, autonomous execution, and cross-functional influence without direct authority.

  [Advanced] Executive Leadership Program (eCornell)
   ↳ Link: https://www.google.com/search?q=Executive+Leadership+Program+eCornell
   ↳ Focus: Develop high-level strategic decision-making, organizational vision, and enterprise technology governance skills.

-----------------------------------------------------------------

📚 Skill: ENTERPRISE SYSTEM ARCHITECTURE
  [Beginner] Software Archite

![Thankyou](./asset/ty.png)